# 🤖 Autonomous ReAct Agent — Demo Notebook

Walkthrough of every layer of the system.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path().resolve().parent))
try:
    from dotenv import load_dotenv; load_dotenv('../.env')
except ImportError:
    pass
print('Environment ready ✅')

## 1. Run the agent end-to-end

In [ ]:
from agent.graph import run_agent

result = run_agent('What is the 17th Fibonacci number? Calculate with Python.')

print('Final answer:', result['final_answer'])
print('Steps taken: ', result['iteration'])

## 2. Inspect the reasoning trace

In [ ]:
icons = {'reason':'🧠','act':'⚡','load_memory':'🗄️','save_memory':'💾'}
for step in result['trace']:
    print(f"Step {step['iteration']} [{step['action']}]")
    print(f"  Thought: {step['thought'][:200]}")
    print(f"  Result:  {step['result'][:200]}")
    print()

## 3. Persistent memory search

In [ ]:
from memory.store import MemoryStore

store = MemoryStore()
print(f'Stored memories: {store.size}')
results = store.search('Fibonacci calculation', top_k=3)
for r in results:
    print(f"  [{r['score']:.2f}] {r['text'][:120]}")

## 4. Run tools directly

In [ ]:
from tools.python_repl import python_repl_tool
from tools.wikipedia import wikipedia_tool

# Python REPL
code = '[x for x in range(2,50) if all(x%i!=0 for i in range(2,x))]'
print('Primes < 50:', python_repl_tool(f'print({code})'))

# Wikipedia
print(wikipedia_tool('FAISS', sentences=3))

## 5. Hit the FastAPI endpoints

> **Requires** `uvicorn api.main:app --port 8000` running in another terminal.

In [ ]:
import httpx, json

try:
    r = httpx.post('http://localhost:8000/agent/run',
                   json={'task': 'Compute factorial of 10 with Python'}, timeout=60)
    data = r.json()
    print('Answer:', data['final_answer'])
    print('Steps: ', data['iterations'])
except Exception as e:
    print(f'Server not running: {e}')
    print('Start with: uvicorn api.main:app --port 8000')

## 6. Stream SSE events

In [ ]:
import httpx, json

try:
    with httpx.stream('POST', 'http://localhost:8000/agent/stream',
                      json={'task': 'Sum of primes under 30 using Python'},
                      timeout=120) as r:
        for line in r.iter_lines():
            if line.startswith('data:'):
                payload = json.loads(line[5:])
                if 'action' in payload:
                    print(f"[{payload['action']}] {payload.get('thought','')[:100]}")
                elif 'final_answer' in payload:
                    print(f"\n✅ {payload['final_answer']}")
except Exception as e:
    print(f'Server not running: {e}')